### **Title:** **Luong Attention (Multiplicative Attention)**

### **Objectives:**

* To implement an RNN Encoder–Decoder architecture with Luong (Multiplicative) Attention for English–French neural machine translation using the PyTorch deep learning framework.  
* To understand the working of the Luong Attention mechanism and how multiplicative alignment scores enable the decoder to focus on the most relevant encoder hidden states during each decoding step.  
* To evaluate the performance of the Luong Attention-based sequence-to-sequence model in improving translation accuracy, contextual understanding, and handling of long input sequences compared to a basic RNN Encoder–Decoder model without attention.

### **Theory:**

This project implements a Recurrent Neural Network (RNN) Encoder–Decoder architecture with Luong (Multiplicative) Attention for English–French neural machine translation using the PyTorch framework. The encoder converts each word in the input English sentence into a dense vector representation through an embedding layer and processes the sequence using a Gated Recurrent Unit (GRU)-based RNN. As the input sequence is processed, the encoder generates a hidden state for every input token, capturing both semantic and contextual information.

Unlike the basic Encoder–Decoder model, which compresses the entire input sentence into a single fixed-length context vector, the Luong Attention mechanism enables the decoder to access all encoder hidden states throughout the translation process. At each decoding step, the decoder first produces its current hidden state and then calculates attention scores by comparing this hidden state with every encoder hidden state using a multiplicative alignment function (dot-product or general score). The attention scores are normalized using the Softmax function to obtain attention weights that indicate the importance of each input word for predicting the current output word.

The attention weights are used to compute a context vector, which is a weighted sum of the encoder hidden states. This context vector is then concatenated with the decoder hidden state and passed through a linear layer to generate the probability distribution over the target vocabulary. By dynamically focusing on the most relevant parts of the source sentence at every decoding step, Luong Attention significantly improves translation quality, particularly for longer and more complex sentences, while remaining computationally efficient compared to additive attention mechanisms.

During training, the model employs teacher forcing, where the correct target word is fed as the next input to the decoder instead of the model's previous prediction. This technique accelerates convergence, improves training stability, and reduces error accumulation during learning. The network parameters are optimized using the Adam optimizer, while the Negative Log Likelihood (NLL) Loss function measures the difference between the predicted and actual target words. Gradients are computed through backpropagation, enabling the model to iteratively update its parameters and learn accurate English–French translations.


In [9]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random
from torch.utils.data import TensorDataset

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


In [10]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [11]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [12]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

In [13]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [14]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [15]:
from google.colab import files

uploaded = files.upload()

Saving eng-fra.txt to eng-fra.txt


In [16]:
PATH = r'eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['elles sont probablement etasuniennes', 'they re probably americans']


15

In [17]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class LuongDotAttention(nn.Module):
    def __init__(self, hidden_size):
        super(LuongDotAttention, self).__init__()

        # For:
        # s~_t = tanh(W_c[c_t;s_t])
        self.Wc = nn.Linear(hidden_size * 2, hidden_size)


    def forward(self, query, keys):
        """
        query:
            Current decoder hidden state s_t
            Shape: (batch_size, 1, hidden_size)

        keys:
            Encoder hidden states h_1,...,h_T
            Shape: (batch_size, seq_len, hidden_size)

        Returns:
            attentional_hidden:
                s~_t
                Shape: (batch_size, 1, hidden_size)

            weights:
                attention weights alpha_t
                Shape: (batch_size, 1, seq_len)
        """

        # Alignment scores:
        # e_{t,i} = s_t^T h_i
        scores = torch.bmm(
            query,
            keys.transpose(1, 2)
        )

        # Attention weights:
        # alpha_{t,i} = softmax(e_{t,i})
        weights = F.softmax(scores, dim=-1)

        # Context vector:
        # c_t = sum(alpha_{t,i} * h_i)
        context = torch.bmm(
            weights,
            keys
        )

        # Concatenate context and decoder hidden state:
        # [c_t ; s_t]
        combined = torch.cat(
            (context, query),
            dim=-1
        )

        # Attentional hidden state:
        # s~_t = tanh(W_c[c_t;s_t])
        attentional_hidden = torch.tanh(
            self.Wc(combined)
        )

        return attentional_hidden, weights

In [19]:
class LuongDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(LuongDecoderRNN, self).__init__()

        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(output_size, hidden_size)

        self.dropout = nn.Dropout(dropout_p)

        self.rnn = nn.RNN(
            hidden_size,
            hidden_size,
            batch_first=True
        )

        self.attention = LuongDotAttention(hidden_size)

        self.concat = nn.Linear(
            hidden_size * 2,
            hidden_size
        )

        self.out = nn.Linear(
            hidden_size,
            output_size
        )

    def forward(self,
                encoder_outputs,
                encoder_hidden,
                target_tensor=None):

        batch_size = encoder_outputs.size(0)

        decoder_input = torch.full(
            (batch_size, 1),
            SOS_token,
            dtype=torch.long,
            device=device
        )

        decoder_hidden = encoder_hidden

        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):

            decoder_output, decoder_hidden, attn = self.forward_step(
                decoder_input,
                decoder_hidden,
                encoder_outputs
            )

            decoder_outputs.append(decoder_output)
            attentions.append(attn)

            if target_tensor is not None:
                decoder_input = target_tensor[:, i].unsqueeze(1)
            else:
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)

        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions

    def forward_step(self,
                     input,
                     hidden,
                     encoder_outputs):

        embedded = self.dropout(
            self.embedding(input)
        )

        # Decoder RNN first
        rnn_output, hidden = self.rnn(
            embedded,
            hidden
        )

        # Prepare query
        query = rnn_output

        # Luong attention
        context, attn_weights = self.attention(
            query,
            encoder_outputs
        )

        # Concatenate context and decoder output
        concat_input = torch.cat(
            (rnn_output, context),
            dim=2
        )

        concat_output = torch.tanh(
            self.concat(concat_input)
        )

        output = self.out(concat_output)

        return output, hidden, attn_weights

In [20]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

In [21]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [22]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [23]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [24]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

In [25]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [26]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [27]:
from torch.utils.data import TensorDataset
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = LuongDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 21s (- 14m 17s) (5 2%) 1.8951
0m 35s (- 11m 21s) (10 5%) 1.2363
0m 49s (- 10m 12s) (15 7%) 0.9857
1m 3s (- 9m 29s) (20 10%) 0.8057
1m 16s (- 8m 57s) (25 12%) 0.6592
1m 30s (- 8m 32s) (30 15%) 0.5216
1m 44s (- 8m 10s) (35 17%) 0.4135
1m 57s (- 7m 51s) (40 20%) 0.3299
2m 12s (- 7m 36s) (45 22%) 0.2636
2m 29s (- 7m 27s) (50 25%) 0.2212
2m 43s (- 7m 12s) (55 27%) 0.1954
2m 57s (- 6m 54s) (60 30%) 0.1730
3m 10s (- 6m 36s) (65 32%) 0.1526
3m 24s (- 6m 19s) (70 35%) 0.1411
3m 38s (- 6m 4s) (75 37%) 0.1315
3m 52s (- 5m 48s) (80 40%) 0.1185
4m 6s (- 5m 33s) (85 42%) 0.1135
4m 20s (- 5m 18s) (90 45%) 0.0996
4m 34s (- 5m 3s) (95 47%) 0.0977
4m 48s (- 4m 48s) (100 50%) 0.0924
5m 2s (- 4m 33s) (105 52%) 0.0935
5m 16s (- 4m 19s) (110 55%) 0.0943
5m 31s (- 4m 5s) (115 57%) 0.0973
5m 46s (- 3m 51s) (120 60%) 0.0889
6m 1s (- 3m 36s) (125 62%) 0.0820
6m 15s (- 3m 22s) (130 65%)

In [28]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> vous etes toutes folles
= you re all mad
< you re all crazy <EOS>

> tu desobeis aux ordres
= you re disobeying orders
< you re disobeying orders <EOS>

> nous sommes trop occupes
= we re too busy
< we re too busy <EOS>

> vous etes deloyale
= you re disloyal
< you are too tense <EOS>

> vous etes beau
= you are beautiful
< you are beautiful <EOS>

> je suis vraiment contente
= i m really happy
< i m really happy <EOS>

> vous etes un traitre
= you re a traitor
< you re a traitor <EOS>

> il est paresseux
= he s lazy
< he is lazy <EOS>

> ils sont encore jeunes
= they re still young
< they re still young <EOS>

> vous etes tres peureuse
= you re very timid
< you re very timid <EOS>



### **Discussion**

The implemented model successfully demonstrates an RNN Encoder–Decoder architecture with Luong (Multiplicative) Attention for English–French machine translation. The encoder processes the input sentence and generates hidden state representations, while the decoder produces the translated output word by word. The Luong Attention mechanism enhances the translation process by calculating multiplicative alignment scores between the decoder hidden state and all encoder hidden states, allowing the decoder to focus on the most relevant parts of the input sequence at each step. After preprocessing the dataset and training the model using teacher forcing, Adam optimizer, and Negative Log Likelihood (NLL) loss, the continuous reduction in training loss shows that the model effectively learned the translation patterns. The attention-based approach improves contextual understanding, handles longer sequences more efficiently, and provides better translation performance compared to the traditional Encoder–Decoder model without attention.  

### **Conclusion**

The RNN Encoder–Decoder model with Luong Attention effectively improves machine translation by enabling the decoder to dynamically focus on the most relevant encoder hidden states during each decoding step. By overcoming the limitations of the fixed-length context vector used in traditional Encoder–Decoder models, the attention mechanism enhances contextual understanding and improves translation accuracy, especially for longer sentences. The successful implementation provides a practical understanding of attention-based sequence-to-sequence models and establishes a foundation for exploring more advanced neural machine translation architectures.
